# Using Pyrea with Nutrimouse data: the new graph API

This notebook follows [Nutrimouse.ipynb](Nutrimouse.ipynb) in the same order:
load the two views; optimize **Parea 1 with hierarchical clustering**; optimize
**Parea 1 with spectral clustering**; evaluate against the three original
`mvlearn` methods; and optimize **Parea 2 with hierarchical clustering**.
Selected configurations and a results table are computed from this run.

The Parea examples use `pyrea.nn`, `pyrea.models`, and `pyrea.optim`, replacing
the older eager helpers. We retain the supplied measurements without feature
standardization and the original search ranges and population/generation counts.
Every final layer has **`k_final=2`**, the known number of Nutrimouse genotype
classes. Intermediate cluster counts remain tunable. Specifying this class count
is prior information; individual genotype labels enter only external evaluation,
never clustering or genetic fitness.

This repeats the experiments and teaching sequence, rather than promising
identical historical partitions: linkage selection has been corrected, spectral
operations now have explicit seeds, and the graph optimizer uses its own
evolutionary implementation. The active parameter differences are described below.

## Imports and installation

From the repository root, install the checkout and notebook tools:

```bash
python -m pip install -e .
python -m pip install -r notebooks/requirements.txt
jupyter lab notebooks/Nutrimouse_New_API.ipynb
```

The bundled CSV files are exactly the data distributed with `mvlearn==0.5.0`,
so Parea needs no download or `mvlearn` installation. To also run the three
optional comparison methods, use a Python 3.11 environment with the tested
numerical stack:

```bash
python -m pip install numpy==1.26.4 scipy==1.11.4 scikit-learn==1.3.2 seaborn==0.13.2
python -m pip install --no-deps mvlearn==0.5.0
```

`mvlearn==0.5.0` declares `matplotlib<=3.3.4`, which predates Python 3.11
wheels. The explicit `--no-deps` command retains the modern matplotlib already
installed for these notebooks. Only the three clustering comparisons are tested
with this combination; it falls outside mvlearn's declared matplotlib range.
If mvlearn is absent, its results are explicitly marked **not run**.
Select the kernel for your environment and **Restart Kernel and Run All**.

In [1]:
from collections import Counter
from pathlib import Path
import importlib.metadata
import warnings

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import adjusted_mutual_info_score as ami_score
from sklearn.metrics import normalized_mutual_info_score as nmi_score
from sklearn.metrics import silhouette_score

from pyrea import models, nn, optim

SEED = 42
K_FINAL = 2
LINKAGES = ['single', 'complete', 'average', 'weighted',
            'centroid', 'median', 'ward', 'ward2']

display(pd.Series({name: importlib.metadata.version(name)
                   for name in ['pyrea', 'numpy', 'scipy', 'scikit-learn', 'networkx']},
                  name='Version'))

pyrea           1.0.12
numpy           1.26.4
scipy           1.11.4
scikit-learn     1.3.2
networkx         3.6.1
Name: Version, dtype: object

## Load data

As in the original notebook, the two aligned views are gene expression and
lipids, and `y` is genotype. Diet is retained as a separate descriptive annotation.
Neither annotation is passed into a graph or fitness callable. Row-based sample
IDs make alignment visible; they are not original biological identifiers.

In [2]:
project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / 'notebooks/data/nutrimouse/gene.csv').is_file()), None)
if project_root is None:
    raise FileNotFoundError('Run this notebook from within the Pyrea checkout.')
data_dir = project_root / 'notebooks/data/nutrimouse'
gene = pd.read_csv(data_dir / 'gene.csv')
lipid = pd.read_csv(data_dir / 'lipid.csv')
y = pd.read_csv(data_dir / 'genotype.csv')['genotype'].to_numpy()
diet = pd.read_csv(data_dir / 'diet.csv')['diet'].to_numpy()
sample_ids = pd.Index([f'mouse_{i:02d}' for i in range(len(gene))], name='sample')
gene.index = lipid.index = sample_ids
inputs = {'gene': gene, 'lipid': lipid}
data = [gene.to_numpy(), lipid.to_numpy()]  # same ordered views for mvlearn

assert gene.shape == (40, 120) and lipid.shape == (40, 21)
assert len(y) == len(diet) == len(gene)
assert len(np.unique(y)) == K_FINAL
assert all(np.isfinite(view).all() for view in data)
print(f'Number of views: {len(data)}')
for i, view in enumerate(data, 1):
    print(f'Shape of view {i}: {view.shape[0]} x {view.shape[1]}')

Number of views: 2
Shape of view 1: 40 x 120
Shape of view 2: 40 x 21


The 40 rows in both views refer to the same mice: 120 gene-expression features
and 21 lipid features. The following helpers use **only graph outputs** for
genetic fitness. Hierarchical examples expose fused disagreement distances;
the spectral example converts its fused agreement affinity to distances using
`AffinityToDistance`, matching the original silhouette convention.

Silhouette on a learned fusion matrix measures the partition's separation in
that matrix. It is distinct from genotype agreement and should not be compared
as if every method used a common, fixed distance matrix.

In [3]:
def fused_silhouette(output):
    labels = output['labels']
    if len(np.unique(labels)) != K_FINAL:
        raise ValueError('Rejecting a partition that does not contain k_final=2 clusters.')
    return float(silhouette_score(output['distance'], labels, metric='precomputed'))


def check_output(output):
    assert output['labels'].shape == (40,)
    assert len(np.unique(output['labels'])) == K_FINAL
    distance = output['distance']
    assert distance.shape == (40, 40) and np.isfinite(distance).all()
    assert np.allclose(distance, distance.T) and np.allclose(np.diag(distance), 0)
    assert (distance >= 0).all()


def search_report(search):
    history = pd.DataFrame(search.history)
    print(f'Best observed fused silhouette: {search.best_score:.6f}')
    print(f'{len(history)} requests; {search.n_evaluations} actual evaluations; '
          f'{int(history.cached.sum())} cache hits; '
          f'{int((history.status == "failed").sum())} failed requests.')
    display(pd.Series(search.best_params, name='Selected overrides'))
    display(pd.DataFrame(search.logbook))
    failures = history.loc[history.status == 'failed', ['generation', 'phase', 'error']]
    if not failures.empty:
        display(failures.drop_duplicates(['phase', 'error']))

# Parea 1

## Hierarchical clustering

Parea 1 has two method-specific branches. Each applies a shared clustering
configuration to both views, fuses their partitions, and reclusters the fused
distances. A second fusion joins the branches before a final clustering with
exactly two clusters.

`models.parea_1()` constructs that DAG from public blocks. The four intermediate
clustering configurations remain independently tunable, as in the older helper.
The graph API also exposes the final block's linkage independently: the old
helper tied this linkage to the first precomputed branch. The final **count**
is fixed in the template and excluded from the search space.

In [4]:
parea1_hierarchical = models.parea_1(
    ['gene', 'lipid'], methods=('ward', 'complete'), n_clusters=2,
    precomputed_methods=('ward', 'complete'), k_final=K_FINAL, return_matrix=True,
)
display(pd.DataFrame([
    {'block': name, **block.get_params()}
    for name, block in parea1_hierarchical.named_blocks()
]).fillna(''))

,block,clusterer,n_clusters,method,normalize
0,branch_1,hierarchical,2.0,ward,
1,branch_2,hierarchical,2.0,complete,
2,fusion_1,,,disagreement,False
3,fusion_2,,,disagreement,False
4,recluster_1,hierarchical,2.0,ward,
5,recluster_2,hierarchical,2.0,complete,
6,fusion,,,disagreement,False
7,final,hierarchical,2.0,ward,


### Optimize the hierarchical parameters with a genetic algorithm

This replaces `parea_1_genetic(data, k_min=2, k_max=5, k_final=2,
n_generations=10, n_population=10)`. Integer bounds are inclusive. We search
the same eight linkages and intermediate counts 2–5, with population 10 and
10 generations. Fitness is fused-distance silhouette, not NMI.

The seed controls evolution; score caching avoids repeated deterministic trials.
The search keeps topology fixed and returns an independent configured graph.
Some degenerate fused matrices or non-monotonic centroid/median linkage trees
can fail to yield the requested cluster count. The fitness rejects outputs
whose actual count is not two; `on_error='penalize'` records unsuccessful
candidates, and their errors are displayed rather than aborting the search.

In [5]:
hierarchical_space = {
    name: {'method': LINKAGES, 'n_clusters': optim.Int(2, 5)}
    for name in ['branch_1', 'branch_2', 'recluster_1', 'recluster_2']
}
hierarchical_space['final'] = {'method': LINKAGES}
assert 'n_clusters' not in hierarchical_space['final']

search_hierarchical = optim.genetic_search(
    parea1_hierarchical, inputs, hierarchical_space, fitness=fused_silhouette,
    population_size=10, generations=10, random_state=SEED,
    on_error='penalize', cache=True,
)
search_report(search_hierarchical)

Best observed fused silhouette: 1.000000
100 requests; 33 actual evaluations; 67 cache hits; 3 failed requests.


branch_1__method           weighted
branch_1__n_clusters              5
branch_2__method             median
branch_2__n_clusters              3
recluster_1__method          median
recluster_1__n_clusters           2
recluster_2__method          median
recluster_2__n_clusters           3
final__method                  ward
Name: Selected overrides, dtype: object

,generation,evaluations,cache_hits,valid,failed,generation_best,best_score,mean_score
0,0,10,0,8,2,0.871536,0.871536,0.700367
1,1,5,4,10,0,0.871536,0.871536,0.695288
2,2,6,3,10,0,0.871536,0.871536,0.793853
3,3,4,5,10,0,0.962500,0.962500,0.856713
4,4,2,7,10,0,0.962500,0.962500,0.894015
5,5,1,8,9,1,0.962500,0.962500,0.952393
6,6,2,7,10,0,0.962500,0.962500,0.948864
7,7,1,8,10,0,0.962500,0.962500,0.939167
8,8,1,8,10,0,1.000000,1.000000,0.966250
9,9,1,8,10,0,1.000000,1.000000,0.975000


,generation,phase,error
3,0,fitness,ValueError: Rejecting a partition that does no...


### Execute the selected graph

The replacement for passing the returned positional parameter list into
`parea_1()` is to execute `best_graph`. It retains `k_final=2`.

In [6]:
output_hierarchical = search_hierarchical.best_graph(**inputs)
check_output(output_hierarchical)
labels_hierarchical = output_hierarchical['labels']
assert search_hierarchical.best_graph.get_params()['final__n_clusters'] == K_FINAL
assert np.isclose(fused_silhouette(output_hierarchical), search_hierarchical.best_score)
print('Predicted labels:', labels_hierarchical)
print('Genotype labels: ', y)

Predicted labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 1 0 0 0
 0 0 1]
Genotype labels:  ['wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt'
 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar']


## Spectral clustering

The original notebook repeats Parea 1 with spectral clustering and **agreement**
fusion. Here its topology is written explicitly as an `nn.Module`, since the
predefined Parea builders use hierarchical clustering and disagreement.

Both feature-level clusterers use nearest-neighbor affinity. Subsequent
clusterers consume precomputed agreement affinities. The last spectral block
has `n_clusters=K_FINAL`; `AffinityToDistance` is a separate output branch used
for silhouette, not as input to spectral clustering.

In [7]:
class SpectralParea1(nn.Module):
    def __init__(self, k_final=2, random_state=42):
        super().__init__()
        self.branch_1 = nn.Cluster('spectral', n_clusters=2,
                                   affinity='nearest_neighbors', n_neighbors=10,
                                   random_state=random_state)
        self.branch_2 = nn.Cluster('spectral', n_clusters=2,
                                   affinity='nearest_neighbors', n_neighbors=10,
                                   random_state=random_state)
        self.fusion_1 = nn.Fusion('agreement')
        self.fusion_2 = nn.Fusion('agreement')
        self.recluster_1 = nn.Cluster('spectral', input_kind=nn.AFFINITY,
                                      n_clusters=2, random_state=random_state)
        self.recluster_2 = nn.Cluster('spectral', input_kind=nn.AFFINITY,
                                      n_clusters=2, random_state=random_state)
        self.fusion = nn.Fusion('agreement')
        self.final = nn.Cluster('spectral', input_kind=nn.AFFINITY,
                                n_clusters=k_final, random_state=random_state)
        self.to_distance = nn.AffinityToDistance()

    def forward(self, gene, lipid):
        first = self.recluster_1(self.fusion_1(self.branch_1(gene), self.branch_1(lipid)))
        second = self.recluster_2(self.fusion_2(self.branch_2(gene), self.branch_2(lipid)))
        affinity = self.fusion(first, second)
        return {'labels': self.final(affinity), 'affinity': affinity,
                'distance': self.to_distance(affinity)}


parea1_spectral = SpectralParea1(k_final=K_FINAL, random_state=SEED).compile()
display(pd.Series(parea1_spectral.get_params(), name='Spectral template'))

branch_1__clusterer                   spectral
branch_1__n_clusters                         2
branch_1__affinity           nearest_neighbors
branch_1__n_neighbors                       10
branch_1__random_state                      42
branch_2__clusterer                   spectral
branch_2__n_clusters                         2
branch_2__affinity           nearest_neighbors
branch_2__n_neighbors                       10
branch_2__random_state                      42
fusion_1__method                     agreement
fusion_1__normalize                      False
fusion_2__method                     agreement
fusion_2__normalize                      False
recluster_1__clusterer                spectral
recluster_1__n_clusters                      2
recluster_1__random_state                   42
recluster_2__clusterer                spectral
recluster_2__n_clusters                      2
recluster_2__random_state                   42
fusion__method                       agreement
fusion__norma

### Optimize the spectral parameters with a genetic algorithm

This replaces `parea_1_genetic_spectral(data, k_min=2, k_max=8, k_final=2,
n_neighbors_min=10, n_neighbors_max=15, n_population=10, n_generations=3)`.
The population, generations, intermediate counts, and neighbor bounds match.
Neighbor counts apply only to the feature-level nearest-neighbor graphs;
they have no effect on blocks receiving precomputed affinity and are therefore
not searched there. All spectral blocks have explicit estimator seeds.

Agreement graphs can be disconnected. Warnings are captured and summarized
below so they remain visible without hundreds of repeated messages. Such
partitions can still receive high internal silhouette; external NMI is separate.

In [8]:
spectral_space = {
    'branch_1': {'n_clusters': optim.Int(2, 8), 'n_neighbors': optim.Int(10, 15)},
    'branch_2': {'n_clusters': optim.Int(2, 8), 'n_neighbors': optim.Int(10, 15)},
    'recluster_1': {'n_clusters': optim.Int(2, 8)},
    'recluster_2': {'n_clusters': optim.Int(2, 8)},
}

with warnings.catch_warnings(record=True) as spectral_warnings:
    warnings.simplefilter('always')
    search_spectral = optim.genetic_search(
        parea1_spectral, inputs, spectral_space, fitness=fused_silhouette,
        population_size=10, generations=3, random_state=SEED,
        on_error='penalize', cache=True,
    )

search_report(search_spectral)
warning_counts = Counter((w.category.__name__, str(w.message)) for w in spectral_warnings)
display(pd.DataFrame([{'category': category, 'message': message, 'count': count}
                      for (category, message), count in warning_counts.items()]))

Best observed fused silhouette: 1.000000
37 requests; 21 actual evaluations; 16 cache hits; 0 failed requests.


branch_1__n_clusters        7
branch_1__n_neighbors      13
branch_2__n_clusters        4
branch_2__n_neighbors      11
recluster_1__n_clusters     2
recluster_2__n_clusters     2
Name: Selected overrides, dtype: int64

,generation,evaluations,cache_hits,valid,failed,generation_best,best_score,mean_score
0,0,10,0,10,0,0.653886,0.653886,0.450656
1,1,6,3,10,0,1.000000,1.000000,0.585543
2,2,3,6,10,0,1.000000,1.000000,0.740440
3,3,2,7,10,0,1.000000,1.000000,0.911686


,category,message,count
0,UserWarning,"Graph is not fully connected, spectral embeddi...",7


### Execute the selected spectral graph

As above, `best_graph` replaces the old parameter-list replay. The affinity
and its explicitly converted distance matrix are both available for inspection.

In [9]:
with warnings.catch_warnings(record=True) as selected_spectral_warnings:
    warnings.simplefilter('always')
    output_spectral = search_spectral.best_graph(**inputs)
check_output(output_spectral)
labels_spectral = output_spectral['labels']
assert search_spectral.best_graph.get_params()['final__n_clusters'] == K_FINAL
assert np.isclose(fused_silhouette(output_spectral), search_spectral.best_score)
print('Predicted labels:', labels_spectral)
print('Genotype labels: ', y)
print('Selected-graph warnings:', dict(Counter(str(w.message) for w in selected_spectral_warnings)))

Predicted labels: [0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 0 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1]
Genotype labels:  ['wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt'
 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar']
Selected-graph warnings: {'Graph is not fully connected, spectral embedding may not work as expected.': 1}


# Evaluation

## Parea 1

As in the original, evaluate the selected hierarchical and spectral partitions
against genotype using normalized mutual information (NMI). Adjusted mutual
information (AMI) is also shown. Labels are arbitrary cluster IDs; neither
metric requires matching their numeric values to the genotype IDs.

In [10]:
s_nmi_hierarchical = nmi_score(y, labels_hierarchical)
s_nmi_spectral = nmi_score(y, labels_spectral)
display(pd.DataFrame([
    {'method': 'Parea 1 (hierarchical)', 'NMI': s_nmi_hierarchical,
     'AMI': ami_score(y, labels_hierarchical)},
    {'method': 'Parea 1 (spectral)', 'NMI': s_nmi_spectral,
     'AMI': ami_score(y, labels_spectral)},
]).set_index('method'))

,NMI,AMI
method,,
Parea 1 (hierarchical),0.147082,0.120905
Parea 1 (spectral),0.684363,0.678275


## mvlearn

We repeat the original three comparison methods on the **same raw views**,
each with two clusters and seed 42. Co-regularized spectral clustering retains
the original `n_init=100`. These methods are not part of the Parea graph API.
If mvlearn is absent, the next three examples show an explicit not-run message
and the final table retains those rows with missing scores.

In [11]:
try:
    from mvlearn.cluster import (
        MultiviewCoRegSpectralClustering, MultiviewSphericalKMeans, MultiviewKMeans,
    )
except ModuleNotFoundError as error:
    if error.name != 'mvlearn':
        raise  # do not disguise a broken mvlearn installation as an absent package
    MVLEARN_AVAILABLE = False
    print('mvlearn comparisons not run: install the optional dependency described above.')
else:
    MVLEARN_AVAILABLE = True
    print('mvlearn version:', importlib.metadata.version('mvlearn'))

comparison_labels = {}

mvlearn version: 0.5.0


### Multiview co-regularized spectral clustering

In [12]:
if MVLEARN_AVAILABLE:
    mv_coreg_spectral = MultiviewCoRegSpectralClustering(
        n_clusters=K_FINAL, random_state=SEED, n_init=100,
    )
    labels_mv_coreg_spectral = mv_coreg_spectral.fit_predict(data)
    comparison_labels['mvlearn (co-regularized spectral)'] = labels_mv_coreg_spectral
    s_nmi_mv_coreg_spectral = nmi_score(y, labels_mv_coreg_spectral)
    print('NMI:', s_nmi_mv_coreg_spectral)
else:
    print('Not run: mvlearn is not installed.')

NMI: 0.4591921489796446


### Multiview spherical KMeans

In [13]:
if MVLEARN_AVAILABLE:
    mv_spherical_kmeans = MultiviewSphericalKMeans(n_clusters=K_FINAL, random_state=SEED)
    labels_spherical_kmeans = mv_spherical_kmeans.fit_predict(data)
    comparison_labels['mvlearn (spherical KMeans)'] = labels_spherical_kmeans
    s_nmi_spherical_kmeans = nmi_score(y, labels_spherical_kmeans)
    print('NMI:', s_nmi_spherical_kmeans)
else:
    print('Not run: mvlearn is not installed.')

NMI: 0.6843628400956305


### Multiview KMeans

In [14]:
if MVLEARN_AVAILABLE:
    mv_kmeans = MultiviewKMeans(n_clusters=K_FINAL, random_state=SEED)
    labels_kmeans = mv_kmeans.fit_predict(data)
    comparison_labels['mvlearn (KMeans)'] = labels_kmeans
    s_nmi_kmeans = nmi_score(y, labels_kmeans)
    print('NMI:', s_nmi_kmeans)
else:
    print('Not run: mvlearn is not installed.')

NMI: 0.5615896365639194


# Parea 2

## Hierarchical clustering and genetic optimization

The original `parea_2_mv_genetic()` example chooses a hierarchical method and
cluster count for each view, fuses their partitions, then reclusters. Here
`models.parea_2()` creates the equivalent **fixed-final-count** variant.

We keep the original eight linkages, intermediate counts 2–6, population 100,
and 5 generations. The requested change is `k_final=2`: a single final
clustering replaces the original strict-consensus endpoint. There is therefore
no second final branch or final-branch cluster count to optimize. We search
the linkage of the one active final block while its count stays fixed at two.

In [15]:
parea2_hierarchical = models.parea_2(
    ['gene', 'lipid'], methods=('ward', 'complete'), n_clusters=2,
    precomputed_methods=('ward',), k_final=K_FINAL, return_matrix=True,
)
parea2_space = {
    'cluster_1': {'method': LINKAGES, 'n_clusters': optim.Int(2, 6)},
    'cluster_2': {'method': LINKAGES, 'n_clusters': optim.Int(2, 6)},
    'final': {'method': LINKAGES},
}
assert 'n_clusters' not in parea2_space['final']

search_parea2 = optim.genetic_search(
    parea2_hierarchical, inputs, parea2_space, fitness=fused_silhouette,
    population_size=100, generations=5, random_state=SEED,
    on_error='penalize', cache=True,
)
search_report(search_parea2)

Best observed fused silhouette: 0.962500
595 requests; 341 actual evaluations; 254 cache hits; 18 failed requests.


cluster_1__method          single
cluster_1__n_clusters           2
cluster_2__method        complete
cluster_2__n_clusters           2
final__method                ward
Name: Selected overrides, dtype: object

,generation,evaluations,cache_hits,valid,failed,generation_best,best_score,mean_score
0,0,99,1,92,8,0.895960,0.895960,0.487878
1,1,68,31,92,8,0.928535,0.928535,0.601251
2,2,55,44,99,1,0.928535,0.928535,0.680701
3,3,54,45,100,0,0.962500,0.962500,0.776849
4,4,39,60,99,1,0.962500,0.962500,0.855965
5,5,26,73,100,0,0.962500,0.962500,0.895429


,generation,phase,error
18,0,fitness,ValueError: Rejecting a partition that does no...


### Execute Parea 2 with the selected configuration

This replaces `parea_2_mv(data, **params_parea2)`, with the final count fixed
both during optimization and when executing the selected graph.

In [16]:
output_parea2 = search_parea2.best_graph(**inputs)
check_output(output_parea2)
labels_parea_2 = output_parea2['labels']
assert search_parea2.best_graph.get_params()['final__n_clusters'] == K_FINAL
assert np.isclose(fused_silhouette(output_parea2), search_parea2.best_score)
s_nmi_parea_2 = nmi_score(y, labels_parea_2)
print('Predicted labels:', labels_parea_2)
print('Genotype labels: ', y)
print('NMI:', s_nmi_parea_2)

Predicted labels: [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 1 1 1 0 0 0 0 0 1 1 0 0 0
 0 0 1]
Genotype labels:  ['wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'wt'
 'wt' 'wt' 'wt' 'wt' 'wt' 'wt' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar' 'ppar'
 'ppar' 'ppar' 'ppar' 'ppar']
NMI: 0.27463724921161164


# Conclusions

Every score below is calculated from **this execution**, replacing the original
notebook's hand-written table of best historical runs. The three Parea methods
use genetic search with fused silhouette fitness; their NMI values describe
external agreement after that selection. All six methods have two output clusters.
No scores from missing optional comparisons are filled in from earlier results.

In [17]:
parea_outputs = {
    'Parea 1 (hierarchical)': output_hierarchical,
    'Parea 1 (spectral)': output_spectral,
    'Parea 2 (hierarchical)': output_parea2,
}
all_labels = {name: output['labels'] for name, output in parea_outputs.items()}
all_labels.update(comparison_labels)
method_order = [*parea_outputs, 'mvlearn (co-regularized spectral)',
                'mvlearn (spherical KMeans)', 'mvlearn (KMeans)']
rows = []
for name in method_order:
    labels = all_labels.get(name)
    if labels is None:
        rows.append({'method': name, 'status': 'not run (mvlearn absent)'})
        continue
    assert labels.shape == y.shape and len(np.unique(labels)) == K_FINAL
    rows.append({
        'method': name, 'status': 'completed', 'clusters': len(np.unique(labels)),
        'NMI_genotype': nmi_score(y, labels), 'AMI_genotype': ami_score(y, labels),
        'NMI_diet': nmi_score(diet, labels),
        'fused_silhouette': (fused_silhouette(parea_outputs[name])
                             if name in parea_outputs else np.nan),
    })
summary = pd.DataFrame(rows).set_index('method')
assert len(summary) == 6
assert np.isfinite(summary.loc[summary.status == 'completed',
                               ['NMI_genotype', 'AMI_genotype', 'NMI_diet']]).all().all()
display(summary.round(6))

,status,clusters,NMI_genotype,AMI_genotype,NMI_diet,fused_silhouette
method,,,,,,
Parea 1 (hierarchical),completed,2,0.147082,0.120905,0.192765,1.0000
Parea 1 (spectral),completed,2,0.684363,0.678275,0.045374,1.0000
Parea 2 (hierarchical),completed,2,0.274637,0.257859,0.211526,0.9625
mvlearn (co-regularized spectral),completed,2,0.459192,0.448852,0.027140,NaN
mvlearn (spherical KMeans),completed,2,0.684363,0.678275,0.018493,NaN
mvlearn (KMeans),completed,2,0.561590,0.552983,0.099835,NaN


## Configurations

The following tables replace the original notebook's saved parameter lists.
They show the complete active clustering configuration of each selected graph,
including the unchanged final count, and the actual genetic-search budgets.
`best_params` above contains just the searched overrides; `get_params()` also
includes fixed settings. `with_params()` reconstructs a separate selected graph.

In [18]:
searches = {
    'Parea 1 (hierarchical)': search_hierarchical,
    'Parea 1 (spectral)': search_spectral,
    'Parea 2 (hierarchical)': search_parea2,
}
templates = {
    'Parea 1 (hierarchical)': parea1_hierarchical,
    'Parea 1 (spectral)': parea1_spectral,
    'Parea 2 (hierarchical)': parea2_hierarchical,
}
budgets = []
for name, search in searches.items():
    print(name)
    display(pd.DataFrame([
        {'block': block_name, **block.get_params()}
        for block_name, block in search.best_graph.named_blocks()
        if isinstance(block, nn.Cluster)
    ]).fillna(''))
    assert search.best_graph.get_params()['final__n_clusters'] == K_FINAL
    assert templates[name].get_params()['final__n_clusters'] == K_FINAL
    assert set(search.best_graph.graph.edges) == set(templates[name].graph.edges)
    assert pd.DataFrame(search.logbook)['best_score'].is_monotonic_increasing
    assert any(row['status'] == 'ok' for row in search.history)
    budgets.append({'method': name, 'generations': len(search.logbook) - 1,
                    'requests': len(search.history), 'evaluations': search.n_evaluations,
                    'cache_hits': sum(row['cached'] for row in search.history)})
display(pd.DataFrame(budgets).set_index('method'))

reconstructed_parea2 = parea2_hierarchical.with_params(**search_parea2.best_params)
assert np.array_equal(reconstructed_parea2(**inputs)['labels'], labels_parea_2)
display(pd.crosstab(pd.Series(y, name='genotype'),
                    pd.Series(labels_parea_2, name='Parea 2 cluster')))
print('Completed all three Parea searches with k_final=2. '
      f'Executed {len(comparison_labels)} of 3 optional mvlearn comparisons.')

Parea 1 (hierarchical)


,block,clusterer,n_clusters,method
0,branch_1,hierarchical,5,weighted
1,branch_2,hierarchical,3,median
2,recluster_1,hierarchical,2,median
3,recluster_2,hierarchical,3,median
4,final,hierarchical,2,ward


Parea 1 (spectral)


,block,clusterer,n_clusters,affinity,n_neighbors,random_state
0,branch_1,spectral,7,nearest_neighbors,13.0,42
1,branch_2,spectral,4,nearest_neighbors,11.0,42
2,recluster_1,spectral,2,,,42
3,recluster_2,spectral,2,,,42
4,final,spectral,2,,,42


Parea 2 (hierarchical)


,block,clusterer,n_clusters,method
0,cluster_1,hierarchical,2,single
1,cluster_2,hierarchical,2,complete
2,final,hierarchical,2,ward


,generations,requests,evaluations,cache_hits
method,,,,
Parea 1 (hierarchical),10,100,33,67
Parea 1 (spectral),3,37,21,16
Parea 2 (hierarchical),5,595,341,254


Parea 2 cluster,0,1
genotype,,
ppar,12,8
wt,20,0


Completed all three Parea searches with k_final=2. Executed 3 of 3 optional mvlearn comparisons.


### Interpretation and next steps

These are fresh, seeded searches with the same population/generation settings
as the original examples; neither the genetic search nor the silhouette
objective guarantees recovery of the best historical NMI. A silhouette near
one on an ensemble-generated distance matrix can coexist with modest genotype
agreement. Repeated seeds and independent data would be needed for a performance
claim. Genotype provides the two-class prior here; diet has five classes and
is reported separately.

For an exact replay of the original Parea 2 saved partition, including the
historical linkage-wrapper behavior and the strict-consensus endpoint, use
[Nutrimouse_Graphs.ipynb](Nutrimouse_Graphs.ipynb). That endpoint can produce
more than two clusters and is intentionally different from this notebook.
For training once and assigning a new sample with `predict()`, see
[Nutrimouse_Inductive.ipynb](Nutrimouse_Inductive.ipynb).

# References

- [Original Nutrimouse notebook](Nutrimouse.ipynb).
- Pfeifer, Bloice and Schimek (2023), *Parea: Multi-view ensemble clustering for
  cancer subtype discovery*. https://doi.org/10.1016/j.jbi.2023.104406
- Martin et al. (2007), *Novel aspects of PPARalpha-mediated regulation of lipid
  and xenobiotic metabolism revealed through a nutrigenomic study*.
  https://doi.org/10.1002/hep.21510
- [mvlearn Nutrimouse example](https://mvlearn.github.io/auto_examples/datasets/plot_nutrimouse.html).
- [Bundled data provenance, checksums, and attribution](data/nutrimouse/README.md).